# 05 Results Visualization

Publication-ready figures and ablation tables for AED-XAI. All figures are saved to `../figures/` as PDF + PNG at 300 DPI.


In [ ]:
import sys
sys.path.insert(0, "..")

import base64
import json
import warnings
import zlib
from pathlib import Path

import matplotlib
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker
import numpy as np
import pandas as pd
from IPython.display import display

RESULTS_DIR = Path("../data/results")
RESULTS_SEARCH_DIRS = [Path("../data/results"), Path("../results")]
FIGURES_DIR = Path("../figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-paper")
matplotlib.rcParams.update({"font.size": 11, "font.family": "serif"})

METHODS = ["GradCAM", "G-CAME", "D-CLOSE", "LIME", "AED-XAI (Ours)"]
METHOD_COLORS = {
    "GradCAM": "#4878CF",
    "G-CAME": "#6ACC65",
    "D-CLOSE": "#D65F5F",
    "LIME": "#B47CC7",
    "AED-XAI (Ours)": "#C4AD66",
}

def save_figure(fig, name):
    fig.savefig(FIGURES_DIR / f"{name}.pdf", dpi=300, bbox_inches="tight")
    fig.savefig(FIGURES_DIR / f"{name}.png", dpi=300, bbox_inches="tight")
    print(f"Saved {name}.pdf + .png")

print(f"Results directory : {RESULTS_DIR.resolve()}")
print(f"Figures directory : {FIGURES_DIR.resolve()}")

## 1. Output Checklist

Notebook 05 reads outputs produced by the current codebase:

- `results/main_results.csv` from notebook 04 or `scripts/generate_figures.py`: main AED-XAI vs fixed-method comparison.
- `results/notebook_detector_compare.csv` from notebook 04 or `scripts/compare_detectors.py`: YOLOX-S vs Faster R-CNN detector generalization.
- `results/ablation_selector_size.csv` and `results/ablation_selector_size_summary.csv` from `scripts/ablation_selector_size.py`: selector training-size ablation.
- `results/cross_domain.csv` from `scripts/run_cross_domain.py`: COCO/VOC/BDD100K/VisDrone/DOTA/OpenImages domain-shift evaluation.
- Optional long-run CSVs (`threshold_ablation.csv`, `selector_ablation.csv`) are consumed if present and skipped cleanly otherwise.


In [ ]:
expected_outputs = {
    "main_results": [Path("../results/main_results.csv"), Path("../data/results/main_results.csv")],
    "detector_comparison": [Path("../results/notebook_detector_compare.csv"), Path("../results/compare_detectors_per_image.csv")],
    "selector_size_ablation": [Path("../results/ablation_selector_size_summary.csv")],
    "threshold_ablation_optional": [Path("../results/threshold_ablation.csv"), Path("../data/results/threshold_ablation.csv")],
    "selector_ablation_optional": [Path("../results/selector_ablation.csv"), Path("../data/results/selector_ablation.csv")],
    "cross_domain": [Path("../results/cross_domain.csv"), Path("../data/results/cross_domain.csv")],
}

required_outputs = {"main_results"}

for name, candidates in expected_outputs.items():
    found = next((p for p in candidates if p.exists()), None)
    if found is not None:
        status = "found"
    elif name in required_outputs:
        status = "missing REQUIRED"
    else:
        status = "missing optional"
    print(f"{name:<30} {status:<18} {found if found is not None else candidates[0]}")


## 2. Load Experiment Results


In [ ]:
def load_experiment_csv(filename, *, required=False):
    """Load an experiment CSV from the configured results directories.

    main_results.csv is required for Table 1 / Figure 1. The ablation CSVs are
    optional because they come from separate long-running experiments. If an
    optional file is missing, the corresponding figure cell will skip cleanly.
    """
    tried = []
    for base in RESULTS_SEARCH_DIRS:
        candidate = base / filename
        tried.append(str(candidate))
        if candidate.exists():
            print(f"Loaded {candidate}")
            return pd.read_csv(candidate)

    message = (
        f"{filename} not found. Searched: {tried}. "
        "Generate it with the matching experiment script/notebook export."
    )
    if required:
        raise FileNotFoundError(message)
    warnings.warn(message)
    return pd.DataFrame()


main_results = load_experiment_csv("main_results.csv", required=True)
threshold_ablation = load_experiment_csv("threshold_ablation.csv")
selector_ablation = load_experiment_csv("selector_ablation.csv")
cross_domain = load_experiment_csv("cross_domain.csv")

PLOT_METHODS = [method for method in METHODS if method in set(main_results["method"].dropna())]
if not PLOT_METHODS:
    raise ValueError("main_results.csv does not contain any recognized method names.")
missing_methods = [method for method in METHODS if method not in PLOT_METHODS]
if missing_methods:
    warnings.warn(
        "main_results.csv is missing methods: " + ", ".join(missing_methods) +
        ". Figures will use available methods only."
    )

for name, df in [
    ("main_results", main_results),
    ("threshold_ablation", threshold_ablation),
    ("selector_ablation", selector_ablation),
    ("cross_domain", cross_domain),
]:
    print(f"\n{name}: shape={df.shape}")
    if df.empty:
        print("  missing or empty; dependent section will be skipped")
    else:
        display(df.head().round(3))


## 3. Table 1: Main Quantitative Comparison


In [ ]:
metric_cols = ["pg", "ebpg", "oa", "sparsity", "composite", "computation_time"]
summary_mean = main_results.groupby("method")[metric_cols].mean().reindex(PLOT_METHODS)
summary_std = main_results.groupby("method")[metric_cols].std().reindex(PLOT_METHODS)

display(summary_mean.round(3))

best_methods = {}
for metric in metric_cols:
    if metric == "computation_time":
        best_methods[metric] = summary_mean[metric].idxmin()
    else:
        best_methods[metric] = summary_mean[metric].idxmax()

formatted_rows = []
for method in PLOT_METHODS:
    row = {"method": method}
    for metric in metric_cols:
        value = summary_mean.loc[method, metric]
        std = summary_std.loc[method, metric]
        cell = f"{value:.3f} ± {std:.3f}"
        if best_methods[metric] == method:
            cell = f"\textbf{{{cell}}}"
        row[metric] = cell
    formatted_rows.append(row)

table_df = pd.DataFrame(formatted_rows).set_index("method")
display(table_df)

print("LaTeX source:")
print(table_df.to_latex(float_format="%.3f", bold_rows=False, escape=False))


## 4. Figure 1: Composite Score Distribution by Method


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
data = [main_results[main_results["method"] == method]["composite"].dropna().values for method in PLOT_METHODS]
parts = ax.violinplot(data, showmeans=False, showmedians=True, showextrema=False)
for body, method in zip(parts["bodies"], PLOT_METHODS):
    body.set_facecolor(METHOD_COLORS[method])
    body.set_edgecolor("black")
    body.set_alpha(0.65)
if "cmedians" in parts:
    parts["cmedians"].set_color("black")

rng = np.random.default_rng(42)
for idx, method in enumerate(PLOT_METHODS, start=1):
    y = main_results[main_results["method"] == method]["composite"].dropna().values
    x = rng.normal(idx, 0.045, size=len(y))
    ax.scatter(x, y, s=3, alpha=0.15, color="black", linewidths=0)

if "AED-XAI (Ours)" in PLOT_METHODS:
    ours_mean = main_results[main_results["method"] == "AED-XAI (Ours)"]["composite"].mean()
    ax.axhline(ours_mean, color=METHOD_COLORS["AED-XAI (Ours)"], linestyle="--", linewidth=1.5, label="Ours mean")
    ax.legend(frameon=False)
ax.set_xticks(np.arange(1, len(PLOT_METHODS) + 1))
ax.set_xticklabels(PLOT_METHODS, rotation=15, ha="right")
ax.set_ylabel("Composite Score")
ax.set_xlabel("Method")
ax.set_title(f"Composite Score Distribution ({main_results['image_id'].nunique()} images)")
ax.set_ylim(0, 1)
plt.tight_layout()
save_figure(fig, "fig1_composite_distribution")
plt.show()


## 5. Figure 2: Threshold Ablation (Table 4)


In [ ]:
if threshold_ablation.empty:
    print("Skipping threshold ablation: threshold_ablation.csv not found.")
else:
    threshold_order = ["fixed_0.3", "fixed_0.5", "fixed_0.7", "percentile_40", "learned"]
    threshold_order = [mode for mode in threshold_order if mode in set(threshold_ablation["threshold_mode"])]
    thr_group = threshold_ablation.groupby("threshold_mode").agg(
        composite_mean=("composite", "mean"),
        composite_std=("composite", "std"),
        refine_calls_mean=("refine_calls", "mean"),
        converged_pct=("converged", "mean"),
    ).reindex(threshold_order)

    fig, ax1 = plt.subplots(1, 1, figsize=(9, 5))
    x = np.arange(len(threshold_order))
    bars = ax1.bar(x, thr_group["composite_mean"], yerr=thr_group["composite_std"], capsize=4, color="#6A8DDB", edgecolor="black")
    for bar, mode in zip(bars, threshold_order):
        if mode in {"percentile_40", "learned"}:
            bar.set_hatch("///")

    best_idx = int(np.nanargmax(thr_group["composite_mean"].values))
    ax1.text(best_idx, thr_group["composite_mean"].iloc[best_idx] + 0.035, "Best", ha="center", va="bottom", fontweight="bold")
    ax1.set_ylabel("Mean Composite Score")
    ax1.set_ylim(0, 1)
    ax1.set_xticks(x)
    ax1.set_xticklabels(threshold_order, rotation=20, ha="right")

    ax2 = ax1.twinx()
    ax2.plot(x, thr_group["refine_calls_mean"], color="#D65F5F", marker="o", linewidth=2, label="Refine calls")
    ax2.set_ylabel("Mean refine calls / image")
    ax2.set_ylim(0, max(3.0, float(thr_group["refine_calls_mean"].max()) + 0.5))
    ax1.set_title("Threshold Ablation: Composite Score and Refinement Calls")
    plt.tight_layout()
    save_figure(fig, "fig2_threshold_ablation")
    plt.show()

    table4 = thr_group.copy()
    table4["mean_composite ± std"] = table4.apply(lambda r: f"{r['composite_mean']:.3f} ± {r['composite_std']:.3f}", axis=1)
    table4["converged_pct"] = table4["converged_pct"] * 100.0
    table4_display = table4[["mean_composite ± std", "refine_calls_mean", "converged_pct"]]
    display(table4_display.round(3))
    print("LaTeX source:")
    print(table4_display.to_latex(float_format="%.3f"))


## 6. Figure 3: XAI Selector Ablation


In [ ]:
if selector_ablation.empty:
    print("Skipping selector ablation: selector_ablation.csv not found.")
else:
    selector_order = ["always_gradcam", "rule_based", "oracle_mlp"]
    selector_order = [name for name in selector_order if name in set(selector_ablation["selector_type"])]
    selector_group = selector_ablation.groupby("selector_type").agg(
        accuracy=("correct", "mean"),
        composite=("composite", "mean"),
        composite_std=("composite", "std"),
    ).reindex(selector_order)
    color_map = {"always_gradcam": "#8A8A8A", "rule_based": "#4878CF", "oracle_mlp": "#C4AD66"}
    selector_colors = [color_map[name] for name in selector_order]

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    axes[0].bar(selector_order, selector_group["accuracy"] * 100.0, color=selector_colors, edgecolor="black")
    axes[0].set_ylabel("Selection Accuracy (%)")
    axes[0].set_ylim(0, 100)
    axes[0].set_title("Oracle Method Match")
    axes[0].tick_params(axis="x", rotation=20)

    axes[1].bar(selector_order, selector_group["composite"], yerr=selector_group["composite_std"], capsize=4, color=selector_colors, edgecolor="black")
    axes[1].set_ylabel("Mean Composite Score")
    axes[1].set_ylim(0, 1)
    axes[1].set_title("Composite Score")
    axes[1].tick_params(axis="x", rotation=20)
    if "oracle_mlp" in selector_group.index:
        oracle_idx = selector_order.index("oracle_mlp")
        axes[1].text(oracle_idx, selector_group.loc["oracle_mlp", "composite"] + 0.05, "* Ours", ha="center", va="bottom", fontweight="bold")
    fig.suptitle("XAI Selector Ablation: Method Selection Accuracy and Composite Score")
    plt.tight_layout()
    save_figure(fig, "fig3_selector_ablation")
    plt.show()

    display(selector_group.round(3))


## 7. Figure 3b: XAI Selector Training-Size Ablation

How much labeled data does the selector actually need? Trains fresh selectors at several sample sizes (from `scripts/ablation_selector_size.py`) and plots held-out accuracy and achieved composite vs training size, with oracle upper bound and random-selector lower bound as reference lines.

In [ ]:
ABLATION_CSV = Path("../results/ablation_selector_size.csv")
ABLATION_SUMMARY_CSV = Path("../results/ablation_selector_size_summary.csv")

if not ABLATION_SUMMARY_CSV.exists():
    warnings.warn(
        f"{ABLATION_SUMMARY_CSV.name} not found. Generate with notebook 04 or:\n"
        "    python scripts/ablation_selector_size.py --sizes 50,100,200,500,1000 --seeds 42,123,456"
    )
else:
    ablation_summary = pd.read_csv(ABLATION_SUMMARY_CSV).sort_values("train_size").reset_index(drop=True)
    ablation_runs = pd.read_csv(ABLATION_CSV) if ABLATION_CSV.exists() else None
    detector_names = (
        list(ablation_summary["detector"].dropna().unique())
        if "detector" in ablation_summary.columns
        else ["selector"]
    )
    detector_colors = {
        "yolox-s": "#C4AD66",
        "fasterrcnn_resnet50_fpn_v2": "#4878CF",
        "selector": METHOD_COLORS["AED-XAI (Ours)"],
    }

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

    for detector_name in detector_names:
        sub = (
            ablation_summary[ablation_summary["detector"] == detector_name]
            if "detector" in ablation_summary.columns
            else ablation_summary
        ).sort_values("train_size")
        sizes = sub["train_size"].to_numpy()
        color = detector_colors.get(detector_name, None)
        label = detector_name.replace("fasterrcnn_resnet50_fpn_v2", "Faster R-CNN")

        axes[0].errorbar(
            sizes,
            sub["test_accuracy_mean"] * 100.0,
            yerr=sub["test_accuracy_std"] * 100.0,
            marker="o", capsize=4, linewidth=2, label=label, color=color,
        )
        axes[1].errorbar(
            sizes,
            sub["achieved_composite_mean"],
            yerr=sub["achieved_composite_std"],
            marker="o", capsize=4, linewidth=2, label=f"{label} selector", color=color,
        )
        axes[1].plot(sizes, sub["oracle_composite_mean"], linestyle="--", color=color, alpha=0.55)
        axes[1].plot(sizes, sub["random_baseline_composite_mean"], linestyle=":", color=color, alpha=0.55)

    all_sizes = sorted(ablation_summary["train_size"].unique())
    for ax in axes:
        ax.set_xscale("log")
        ax.set_xticks(all_sizes)
        ax.get_xaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())
        ax.grid(True, alpha=0.3)

    axes[0].set_xlabel("Training samples")
    axes[0].set_ylabel("Held-out test accuracy (%)")
    axes[0].set_title("Selector accuracy vs training size")
    axes[0].legend(fontsize=8)

    axes[1].set_xlabel("Training samples")
    axes[1].set_ylabel("Mean composite on test set")
    axes[1].set_title("Achieved composite vs training size")
    axes[1].legend(fontsize=8, loc="lower right")

    fig.suptitle("XAI Selector: Training-Set Size Ablation by Detector")
    plt.tight_layout()
    save_figure(fig, "fig3b_selector_size_ablation")
    plt.show()

    display(ablation_summary.round(4))


## 8. Figure 3c: Detector Generalization Ablation

Compare the full AED-XAI loop with YOLOX-S and Faster R-CNN on the same evaluation images. This uses `results/notebook_detector_compare.csv` exported by notebook 04, or `results/compare_detectors_per_image.csv` from `scripts/compare_detectors.py`.


In [ ]:
detector_candidates = [
    Path("../results/notebook_detector_compare.csv"),
    Path("../results/compare_detectors_per_image.csv"),
    Path("../data/results/notebook_detector_compare.csv"),
    Path("../data/results/compare_detectors_per_image.csv"),
]
detector_path = next((path for path in detector_candidates if path.exists()), None)

if detector_path is None:
    warnings.warn(
        "Detector comparison CSV not found. Run notebook 04's detector-comparison section "
        "or scripts/compare_detectors.py to generate it."
    )
else:
    detector_compare = pd.read_csv(detector_path)
    print(f"Loaded detector comparison from {detector_path}")

    if "has_error" in detector_compare.columns:
        valid_detector = detector_compare[~detector_compare["has_error"].fillna(False).astype(bool)].copy()
    else:
        valid_detector = detector_compare.copy()

    required_cols = {"detector", "composite_score"}
    if not required_cols.issubset(valid_detector.columns):
        warnings.warn(f"Detector comparison CSV missing required columns: {sorted(required_cols - set(valid_detector.columns))}")
    elif valid_detector.empty:
        warnings.warn("Detector comparison CSV has no successful rows to plot.")
    else:
        metrics = [col for col in ["composite_score", "mean_ebpg", "mean_oa", "mean_sparsity", "total_time"] if col in valid_detector.columns]
        detector_summary = valid_detector.groupby("detector")[metrics].agg(["mean", "std"]).round(4)
        display(detector_summary)

        detectors = list(valid_detector["detector"].dropna().unique())
        fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))

        comp = valid_detector.groupby("detector")["composite_score"].agg(["mean", "std"]).reindex(detectors)
        axes[0].bar(detectors, comp["mean"], yerr=comp["std"], capsize=4, color="#C4AD66", edgecolor="black")
        axes[0].set_ylim(0, 1)
        axes[0].set_ylabel("Composite Score")
        axes[0].set_title("Composite by Detector")
        axes[0].tick_params(axis="x", rotation=15)

        if "total_time" in valid_detector.columns:
            runtime = valid_detector.groupby("detector")["total_time"].agg(["mean", "std"]).reindex(detectors)
            axes[1].bar(detectors, runtime["mean"], yerr=runtime["std"], capsize=4, color="#4878CF", edgecolor="black")
            axes[1].set_ylabel("Seconds / Image")
            axes[1].set_title("Runtime by Detector")
            axes[1].tick_params(axis="x", rotation=15)
        else:
            axes[1].axis("off")
            axes[1].text(0.5, 0.5, "No total_time column", ha="center", va="center")

        fig.suptitle("Detector Generalization Ablation")
        plt.tight_layout()
        save_figure(fig, "fig3c_detector_comparison")
        plt.show()


## 9. Figure 4: Performance by Scene Complexity


In [ ]:
complexity_order = ["low", "medium", "high"]
complexity_colors = {"low": "#6ACC65", "medium": "#EEA236", "high": "#D65F5F"}
ours_df = main_results[main_results["method"] == "AED-XAI (Ours)"]
available_complexities = [name for name in complexity_order if name in set(ours_df.get("scene_complexity", pd.Series(dtype=str)).dropna())]

if ours_df.empty or not available_complexities:
    print("Skipping scene-complexity figure: main_results.csv has no AED-XAI rows with low/medium/high scene_complexity.")
else:
    complexity_group = ours_df.groupby("scene_complexity")[["pg", "oa", "composite"]].mean().reindex(available_complexities)

    fig, ax = plt.subplots(1, 1, figsize=(8, 5))
    x = np.arange(len(available_complexities))
    width = 0.22
    for idx, metric in enumerate(["pg", "oa", "composite"]):
        values = complexity_group[metric].values
        bars = ax.bar(x + (idx - 1) * width, values, width=width, label=metric.upper(), edgecolor="black")
        for bar, complexity in zip(bars, available_complexities):
            bar.set_facecolor(complexity_colors[complexity])
            bar.set_alpha(0.6 + idx * 0.12)

    ax.set_xticks(x)
    ax.set_xticklabels(available_complexities)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Score")
    ax.set_title("AED-XAI Performance by Scene Complexity")
    ax.legend(frameon=False)
    plt.tight_layout()
    save_figure(fig, "fig4_scene_complexity")
    plt.show()

    display(complexity_group.round(3))


## 10. Figure 5: Cross-Domain Generalization


In [ ]:
if cross_domain.empty:
    print("Skipping cross-domain figure: cross_domain.csv not found.")
else:
    preferred_order = ["COCO", "VOC", "BDD100K", "VisDrone", "DOTA", "OpenImages"]
    present_domains = list(cross_domain["domain"].dropna().unique())
    domain_order = [domain for domain in preferred_order if domain in present_domains]
    domain_order += [domain for domain in present_domains if domain not in domain_order]
    heatmap_df = cross_domain.groupby(["domain", "method"])["composite"].mean().unstack("method").reindex(index=domain_order, columns=PLOT_METHODS)

    fig, ax = plt.subplots(1, 1, figsize=(10, 4.5))
    im = ax.imshow(heatmap_df.values, cmap="YlOrRd", vmin=0.45, vmax=0.80)
    ax.set_xticks(np.arange(len(PLOT_METHODS)))
    ax.set_xticklabels(PLOT_METHODS, rotation=20, ha="right")
    ax.set_yticks(np.arange(len(domain_order)))
    ax.set_yticklabels(domain_order)
    for i in range(heatmap_df.shape[0]):
        for j in range(heatmap_df.shape[1]):
            value = heatmap_df.iloc[i, j]
            label = "NA" if pd.isna(value) else f"{value:.3f}"
            ax.text(j, i, label, ha="center", va="center", color="black")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Mean Composite Score")
    ax.set_title("Cross-Domain Generalization (Composite Score)")
    plt.tight_layout()
    save_figure(fig, "fig5_cross_domain")
    plt.show()

    display(heatmap_df.round(3))


## 11. Figure 6: Qualitative Comparison


In [ ]:
XAI_CACHE = Path("../data/xai_comparison_cache.json")

def decode_saliency(encoded):
    payload = base64.b64decode(encoded["data"].encode("ascii"))
    raw = zlib.decompress(payload)
    return np.frombuffer(raw, dtype=np.float16).reshape(encoded["shape"]).astype(np.float32)

if not XAI_CACHE.exists():
    print("⚠ Run notebook 03 first to generate xai_comparison_cache.json")
else:
    with XAI_CACHE.open("r", encoding="utf-8") as handle:
        xai_records = json.load(handle)

    scored_records = []
    for record in xai_records:
        scores = {item["method"]: item["composite"] for item in record["methods"]}
        oracle = max(scores, key=scores.get)
        margin = sorted(scores.values(), reverse=True)[0] - sorted(scores.values(), reverse=True)[1]
        scored_records.append((record, oracle, margin, max(scores.values())))

    winners = sorted(scored_records, key=lambda item: item[2], reverse=True)[:2]
    failure = sorted(scored_records, key=lambda item: item[3])[0:1]
    selected = winners + failure

    fig = plt.figure(figsize=(16, 3.6 * len(selected)))
    gs = gridspec.GridSpec(len(selected), 5, figure=fig)
    for row_idx, (record, oracle, _, _) in enumerate(selected):
        image_path = Path("../data/coco/val2017") / record["filename"]
        if image_path.exists():
            image = plt.imread(image_path)
        else:
            image = np.ones((320, 480, 3), dtype=np.float32)
        det = record.get("detection", {})
        bbox = det.get("bbox", [0, 0, image.shape[1] - 1, image.shape[0] - 1])
        x1, y1, x2, y2 = bbox
        method_results = {item["method"]: item for item in record["methods"]}

        ax = fig.add_subplot(gs[row_idx, 0])
        ax.imshow(image)
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor="lime", facecolor="none"))
        ax.set_title(f"{record['filename']}\noracle={oracle}", color="green")
        ax.axis("off")

        for col_idx, method in enumerate(["gradcam", "gcame", "dclose", "lime"], start=1):
            item = method_results[method]
            saliency = decode_saliency(item["saliency"])
            ax = fig.add_subplot(gs[row_idx, col_idx])
            ax.imshow(image)
            ax.imshow(saliency, cmap="jet", alpha=0.5, vmin=0, vmax=1)
            ax.set_title(f"{method}\nComp={item['composite']:.2f}")
            ax.axis("off")

    plt.tight_layout()
    save_figure(fig, "fig6_qualitative")
    plt.show()

## 12. Summary: All Figures Generated


In [ ]:
print("Files in figures directory:")
for path in sorted(FIGURES_DIR.glob("*")):
    size_kb = path.stat().st_size / 1024.0
    print(f"  {path.name:<36} {size_kb:8.1f} KB")

for fig_name in [
    "fig1_composite_distribution",
    "fig2_threshold_ablation",
    "fig3_selector_ablation",
    "fig3b_selector_size_ablation",
    "fig3c_detector_comparison",
    "fig4_scene_complexity",
    "fig5_cross_domain",
    "fig6_qualitative",
]:
    path = FIGURES_DIR / f"{fig_name}.pdf"
    status = "✅" if path.exists() else "❌"
    print(f"  {status} {fig_name}.pdf")